# Linear public goods game with LLM players

Three LLMs (Claude, OpenAI, Grok) play a standard repeated linear public goods game for 10 periods.
Each period every player gets an endowment $E$ and simultaneously chooses a contribution $c_i \in [0, E]$. Payoff:

$$\pi_i = E - c_i + m \sum_j c_j, \qquad 1/n < m < 1$$

With $n=3$ and $m=0.5$, contributing nothing is the dominant strategy, while full contribution maximizes the group payoff.
After each period players only learn the **sum** of all contributions (not individual ones).

Setup: copy `.env.example` to `.env` and add your API keys.

In [ ]:
# %pip install -q anthropic openai python-dotenv pandas matplotlib
import json, os, re
from concurrent.futures import ThreadPoolExecutor

import anthropic
import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

# Game parameters
N_PERIODS = 10
ENDOWMENT = 20
MPCR = 0.5  # marginal per capita return

# Model names are placeholders: set to the models you have access to
PLAYERS = {
    "Claude": {"provider": "anthropic", "model": "claude-sonnet-4-5"},
    "OpenAI": {"provider": "openai", "model": "gpt-4o"},
    "Grok": {"provider": "xai", "model": "grok-4"},
}
N = len(PLAYERS)

clients = {
    "anthropic": anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"]),
    "openai": OpenAI(api_key=os.environ["OPENAI_API_KEY"]),
    "xai": OpenAI(api_key=os.environ["XAI_API_KEY"], base_url="https://api.x.ai/v1"),
}

In [ ]:
INSTRUCTIONS = f"""You are participating in an economic experiment with {N} players (including you) over {N_PERIODS} periods.
In each period every player receives {ENDOWMENT} tokens and simultaneously decides how many tokens (an integer from 0 to {ENDOWMENT}) to contribute to a group project.
Tokens not contributed are kept. Each token in the project returns {MPCR} tokens to EVERY player, regardless of who contributed it.
Your period payoff = {ENDOWMENT} - your contribution + {MPCR} x (sum of all players' contributions).
After each period you learn the total contributions of the group, but not individual contributions.
Your goal is to maximize your total payoff over all periods."""


def build_prompt(history, own_contribs, period):
    lines = [INSTRUCTIONS, ""]
    for t, (total, own) in enumerate(zip(history, own_contribs), 1):
        payoff = ENDOWMENT - own + MPCR * total
        lines.append(f"Period {t}: you contributed {own}, group total was {total}, your payoff was {payoff:.1f}.")
    lines.append(f"\nPeriod {period} of {N_PERIODS}. How many tokens do you contribute?")
    lines.append('Answer only with JSON: {"contribution": <integer>}')
    return "\n".join(lines)


def ask(provider, model, prompt):
    if provider == "anthropic":
        r = clients[provider].messages.create(
            model=model, max_tokens=200, messages=[{"role": "user", "content": prompt}]
        )
        return r.content[0].text
    r = clients[provider].chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}]
    )
    return r.choices[0].message.content


def parse(text):
    m = re.search(r'"contribution"\s*:\s*(\d+)', text) or re.search(r"\d+", text)
    value = int(m.group(1) if m.lastindex else m.group(0))
    return min(max(value, 0), ENDOWMENT)

In [ ]:
history = []  # group totals per period (the only public information)
own = {name: [] for name in PLAYERS}
raw = []

for period in range(1, N_PERIODS + 1):
    prompts = {name: build_prompt(history, own[name], period) for name in PLAYERS}
    # Simultaneous decisions: all players are queried in parallel
    with ThreadPoolExecutor(max_workers=N) as pool:
        futures = {n: pool.submit(ask, PLAYERS[n]["provider"], PLAYERS[n]["model"], prompts[n]) for n in PLAYERS}
        answers = {n: f.result() for n, f in futures.items()}
    choices = {n: parse(a) for n, a in answers.items()}
    for n in PLAYERS:
        own[n].append(choices[n])
        raw.append({"period": period, "player": n, "answer": answers[n]})
    history.append(sum(choices.values()))
    print(f"Period {period}: {choices} -> total {history[-1]}")

df = pd.DataFrame(own, index=range(1, N_PERIODS + 1)).rename_axis("period")
df.to_csv("results.csv")
pd.DataFrame(raw).to_csv("raw_answers.csv", index=False)

In [ ]:
payoffs = ENDOWMENT - df + MPCR * df.sum(axis=1).values[:, None]
print("Total payoffs:\n", payoffs.sum().round(1).to_string())

ax = df.plot(marker="o", figsize=(8, 4.5))
df.mean(axis=1).plot(ax=ax, color="k", ls="--", label="Mean")
ax.set(xlabel="Period", ylabel="Contribution", ylim=(0, ENDOWMENT), title="Contributions over time")
ax.legend()
plt.tight_layout()
plt.savefig("contributions.png", dpi=150)
plt.show()